고객 이탈 여부 예측을 위한 딥러닝 분류 모델 설계

In [7]:
import pandas as pd
from tensorflow.keras.models import Sequential # 순차적 모델 생성을 위한 모듈
from tensorflow.keras.layers import Dense, Dropout # 밀집층(Fully Connected) 및 과적합 방지층 모듈
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ==========================================
# 1. 데이터 로드 및 전처리 (Data Preparation)
# ==========================================

# 균형 데이터셋 파일 읽기
df = pd.read_csv('customer_data_balanced.csv')

# 범주형 변수(ContractType)를 수치형으로 변환하는 원-핫 인코딩 수행
df = pd.get_dummies(df, columns=['ContractType'], dtype=int)

# 독립변수(X: 피처)와 종속변수(y: 이탈 여부 타겟) 분리
X = df.drop(columns=['IsChurn'])
y = df['IsChurn']

# 클래스 불균형 및 데이터 분포를 고려하여 학습(80%) 및 테스트(20%) 데이터셋 분할 (stratify 적용)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# 특성 간의 단위를 맞추기 위한 표준화(StandardScaler) 정규화 적용 (데이터 누수 방지)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # 학습 데이터로 기준을 fit하고 변환
X_test_scaled = scaler.transform(X_test)       # 테스트 데이터는 학습된 기준으로 변환만 수행

# ==========================================
# 2. 딥러닝 모델 설계 및 컴파일 (MLP Model)
# ==========================================

# MLP 기반의 다층 인공신경망 이진 분류 모델 구조 정의
model = Sequential([
    # 입력층 및 첫 번째 은닉층: 입력 피처 개수를 동적으로 설정 (활성화 함수: ReLU)
    Dense(16, activation='relu', input_shape=(X_test_scaled.shape[1],)),
    
    # 과적합(Overfitting) 방지를 위해 뉴런의 20%를 무작위로 끄는 Dropout 레이어 추가
    Dropout(0.2),
    
    # 두 번째 은닉층
    Dense(8, activation='relu'),
    
    # 최종 출력층: 이탈 확률(0~1) 출력을 위해 노드 수 1개 및 Sigmoid 활성화 함수 지정
    Dense(1, activation='sigmoid')
])

# 모델 학습을 위한 손실 함수(Binary Crossentropy) 및 최적화 알고리즘(Adam) 설정
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ==========================================
# 3. 모델 학습 (Model Training)
# ==========================================

# 학습 데이터를 사용해 모델 훈련 수행 (검증 데이터 비율 20% 설정, 10회 반복)
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    verbose=0 # 0(생략),1(학습과정 보여줌),2(진행 상황만 보여줌)
)

# ==========================================
# 4. 모델 성능 평가 및 검증 (Evaluation)
# ==========================================

# 테스트 데이터셋을 이용한 기본 손실(Loss) 및 정확도(Accuracy) 평가
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test)
print(f"테스트손실: {test_loss:.4f}, 테스트정확도: {test_accuracy:.4f}")

# 테스트 데이터 전체에 대한 예측 결과(확률값) 생성
predictions = model.predict(X_test_scaled)
print("예측 결과(상위 5개 확률값):", predictions[:5])

# 시그모이드 출력 확률값을 임계치 0.5 기준으로 이진 클래스(0 또는 1)로 변환
y_pred = (predictions > 0.5).astype(int)

# 요구사항 지표 1: Accuracy(정확도) 계산
acs = accuracy_score(y_test, y_pred)

# 요구사항 지표 2: F1-Score(정밀도와 재현율의 조화평균) 계산
f1 = f1_score(y_test, y_pred)

# 요구사항 지표 3: 정밀도, 재현율 등이 포함된 종합 분류 지표 리포트 생성
cr = classification_report(y_test, y_pred)

# 요구사항 지표 4: 혼동 행렬(정답/오답 분포) 생성
cm = confusion_matrix(y_test, y_pred)

# 최종 검증 성능 지표 화면 출력
print("\n" + "="*50)
print(f"Accuracy (정확도): {acs:.4f}")
print(f"F1-Score (조화평균): {f1:.4f}")
print("="*50)

# [수정 및 추가된 부분] 혼동 행렬 시각화 해석 출력
print("\n[Confusion Matrix (혼동 행렬) 결과]")
print(cm)

# 2x2 행렬에서 각각의 값을 추출 (TN, FP, FN, TP)
tn, fp, fn, tp = cm.ravel()

print("\n[혼동 행렬 상세 해석]")
print(f"• TN (True Negative)  : {tn}명 -> 이탈 안 함(0) 예측 성공 (실제 유지 고객)")
print(f"• FP (False Positive) : {fp}명 -> 이탈 함(1) 예측 실패 (유지 고객을 이탈로 오해)")
print(f"• FN (False Negative) : {fn}명 -> 이탈 안 함(0) 예측 실패 (이탈 고객을 놓쳐서 기업에 가장 치명적!)")
print(f"• TP (True Positive)  : {tp}명 -> 이탈 함(1) 예측 성공 (실제 이탈 고객 방어 가능)")

print("\n[Classification Report]")
print(cr)
print("="*50)



c:\GenAI_ML_DL\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6800 - loss: 0.6321 
테스트손실: 0.6321, 테스트정확도: 0.6800
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
예측 결과(상위 5개 확률값): [[0.6836256 ]
 [0.4455462 ]
 [0.33408567]
 [0.36855257]
 [0.52814853]]

Accuracy (정확도): 0.6800
F1-Score (조화평균): 0.4286

[Confusion Matrix (혼동 행렬) 결과]
[[224  24]
 [104  48]]

[혼동 행렬 상세 해석]
• TN (True Negative)  : 224명 -> 이탈 안 함(0) 예측 성공 (실제 유지 고객)
• FP (False Positive) : 24명 -> 이탈 함(1) 예측 실패 (유지 고객을 이탈로 오해)
• FN (False Negative) : 104명 -> 이탈 안 함(0) 예측 실패 (이탈 고객을 놓쳐서 기업에 가장 치명적!)
• TP (True Positive)  : 48명 -> 이탈 함(1) 예측 성공 (실제 이탈 고객 방어 가능)

[Classification Report]
              precision    recall  f1-score   support

           0       0.68      0.90      0.78       248
           1       0.67      0.32      0.43       152

    accuracy                           0.68       400
   macro avg       0.67      0.61      0.60       400
weighted avg       0.68      0.68      0.65       400



In [ ]:
#!pip install tensorflow

   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
    --------------------------------------- 7.3/350.9 MB 41.2 MB/s eta 0:00:09
   - -------------------------------------- 15.5/350.9 MB 38.9 MB/s eta 0:00:09
   -- ------------------------------------- 23.3/350.9 MB 37.8 MB/s eta 0:00:09
   --- ------------------------------------ 31.5/350.9 MB 37.7 MB/s eta 0:00:09
   ---- ----------------------------------- 39.6/350.9 MB 37.6 MB/s eta 0:00:09
   ----- ---------------------------------- 48.0/350.9 MB 38.2 MB/s eta 0:00:08
   ------ --------------------------------- 55.3/350.9 MB 37.5 MB/s eta 0:00:08
   ------- -------------------------------- 63.2/350.9 MB 37.6 MB/s eta 0:00:08
   -------- ------------------------------- 71.8/350.9 MB 37.8 MB/s eta 0:00:08
   --------- ------------------------------ 80.2/350.9 MB 37.9 MB/s eta 0:00:08
   ---------- ----------------------------- 88.9/350.9 MB 38.1 MB/s eta 0:00:07
   ----------- ---------------------------- 97.3/3


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
